## Constructors Data transform Bronze to Silver

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/03-Silver_Helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

In [0]:
from pyspark.sql import functions as f

In [0]:
# To use reader api options 
constructors_df = spark.read.table(bronze_table).filter((f.col("batch_id") == v_batch_id))

# constructors_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
constructors_selected_df = constructors_df.select(
    f.col("constructorId"),
    f.col("name"),
    f.col("nationality"),
    f.col("ingestion_timestamp"),
    f.col("source_file"),
    f.col("batch_id")
)

In [0]:
# For column rename we can use ALIAS funcrion or also the below function

column_rename = {
    "constructorId" : "constructor_Id",
    "name" : "constructor_name"
    }

constructors_renamed_df = constructors_selected_df.withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

constructors_valid_df = constructors_renamed_df.dropDuplicates(["constructor_Id"]).filter(f.col("constructor_Id").isNotNull())

In [0]:
constructors_final_df = constructors_valid_df.withColumn("nationality", f.initcap("nationality"))

In [0]:
columns = [
    "constructor_name",
    "nationality",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(constructors_final_df, silver_table, "trg.constructor_Id = src.constructor_Id", columns)